In [ ]:
%pip install pandas numpy cv2 h5py matplotlib torch torchvision -q


[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import cv2
import h5py
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

ImportError: libGL.so.1: cannot open shared object file: No such file or directory

In [ ]:
# Resize the image to the target size
def resize_image(img_tensor, target_size=(224, 224)):
    """
    Resize the input image tensor to the target size.
    
    Args:
        img_tensor: torch.Tensor (H, W), the input image.
        target_size: tuple, the desired output size (height, width).
        
    Returns:
        torch.Tensor: The resized image.
    """
    # Convert to numpy array
    img = img_tensor.detach().cpu().numpy()

    # Resize using OpenCV
    resized_img = cv2.resize(img, target_size)

    # Convert back to tensor and normalize to [0, 1]
    return torch.from_numpy(resized_img).float() / 255.0


In [ ]:
df=pd.read_csv("/kaggle/input/brats2020-training-data/BraTS20 Training Metadata.csv")
df.head()

In [ ]:
df.info()
print("Target: ",df['target'].unique())
print("Volume: ",df['volume'].unique().max())#369 patients

In [ ]:
print(sorted(df['slice'].unique()))#3D-->(240x240x155)---->2D-->155x(240x240)

In [ ]:
print("Label1: ",df['label0_pxl_cnt'].max())#Label-1
print("Label2: ",df['label1_pxl_cnt'].max())#Label-2
print("Label4: ",df['label2_pxl_cnt'].max())#Label-4
df[df['label0_pxl_cnt'] != 0].head()

In [ ]:
h5_file_path = '/kaggle/input/brats2020-training-data/BraTS2020_training_data/content/data/volume_41_slice_68.h5'

with h5py.File(h5_file_path, 'r') as f:
    def print_structure(name, obj):
        print(name)
    f.visititems(print_structure)

In [ ]:
with h5py.File(h5_file_path, 'r') as f:
    image_shape = f['image'].shape
    mask_shape = f['mask'].shape

    print("Image shape:", image_shape)
    print("Mask shape:", mask_shape)

In [ ]:
# Load the file
with h5py.File(h5_file_path, 'r') as f:
    image = f['image'][:]  # Shape: (240, 240, 4)
    mask = f['mask'][:]    # Shape: (240, 240, 3)


# --------- Plot the 4 MRI modalities ---------
modality_names = ['T1', 'T1Gd', 'T2', 'T2-FLAIR']
plt.figure(figsize=(12, 4))

for i in range(4):
    plt.subplot(1, 4, i + 1)
    plt.imshow(image[:, :, i], cmap='gray')
    plt.title(modality_names[i])
    plt.axis('off')

plt.suptitle("MRI Modalities", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# --------- Plot the 3 mask channels (one-hot) ---------
mask_labels = ['Label 1 (NCR/NET)', 'Label 2 (ED)', 'Label 4 (ET)']
plt.figure(figsize=(10, 3))

for i in range(3):
    plt.subplot(1, 3, i + 1)
    plt.imshow(mask[:, :, i], cmap='Reds')
    plt.title(mask_labels[i])
    plt.axis('off')

plt.suptitle("Segmentation Mask Channels", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# --------- Overlay Example: Mask on MRI modality ---------
plt.figure(figsize=(3, 3))
plt.imshow(image[:, :, 0], cmap='gray')         # T1
plt.imshow(mask[:, :, 2], cmap='jet', alpha=0.5)  # Label 4 (ET)
plt.title("T1 with Enhancing Tumor Overlay")
plt.axis('off')
plt.show()

In [ ]:
# Adaptive Median Normalization function
def adaptive_median_normalization(channel_tensor, kernel_size=3):
    """
    Apply adaptive median normalization to a single-channel image tensor.

    Input:
      channel_tensor: torch.Tensor with shape (H, W), expected to be float or uint8.
                      Values should be compatible with uint8 range [0, 255].
      kernel_size: int, size of the median filter kernel (odd integer, default=3).

    Returns:
      torch.Tensor, float, normalized to [0, 1].
    """
    img = channel_tensor.numpy().astype(np.uint8)
    filtered = cv2.medianBlur(img, kernel_size)
    normalized = (filtered - filtered.min()) / (filtered.max() - filtered.min() + 1e-8)
    return torch.from_numpy(normalized).float()

# Contrast Limited Adaptive Histogram Equalization (CLAHE)
def clahe(img_tensor, clip_limit=2.0, tile_grid_size=(16, 16)):
    """
    Apply CLAHE to a grayscale image represented as a torch tensor.

    Input:
      img_tensor: torch.Tensor with shape (H, W) or (1, H, W),
                  expected to be float and normalized in [0,1] or uint8 [0,255].
    Returns:
      torch.Tensor, float, normalized to [0,1].
    """
    img = img_tensor.detach().cpu().numpy()

    if img.dtype != np.uint8:
        img = (img * 255).astype(np.uint8)

    if img.ndim == 3 and img.shape[0] == 1:
        img = img.squeeze(0)
    elif img.ndim == 3 and img.shape[2] == 1:
        img = img[:, :, 0]

    clahe_obj = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    clahe_img = clahe_obj.apply(img)

    clahe_tensor = torch.from_numpy(clahe_img).float() / 255.0
    return clahe_tensor

# Bi-lateral filtering
def bilateral_filter(img_tensor, d=9, sigma_color=75, sigma_space=75):
    """
    Apply bilateral filter to a grayscale image tensor.

    Args:
        img_tensor: torch.Tensor (H,W) or (1,H,W), float normalized [0,1] or uint8 [0,255].
        d: Diameter of each pixel neighborhood.
        sigma_color: Filter sigma in the color space.
        sigma_space: Filter sigma in the coordinate space.

    Returns:
        torch.Tensor, filtered image normalized [0,1].
    """
    img = img_tensor.detach().cpu().numpy()

    if img.dtype != np.uint8:
        img = (img * 255).astype(np.uint8)

    if img.ndim == 3 and img.shape[0] == 1:
        img = img.squeeze(0)
    elif img.ndim == 3 and img.shape[2] == 1:
        img = img[:, :, 0]

    filtered_img = cv2.bilateralFilter(img, d=d, sigmaColor=sigma_color, sigmaSpace=sigma_space)
    filtered_tensor = torch.from_numpy(filtered_img).float() / 255.0
    return filtered_tensor

# Morphological sharpening using top-hat transform
def morphological_sharpening(img_tensor, kernel_size=None, sharpening_factor=0.5):
    """
    Apply enhanced morphological sharpening using top-hat transform to a grayscale image tensor.
    Args:
        img_tensor: torch.Tensor (H,W) or (1,H,W), float normalized [0,1] or uint8 [0,255].
        kernel_size: int or None, size of the structuring element (odd integer). If None, 
                     computed as 1/50th of the smaller image dimension (min 3, max 7).
        sharpening_factor: float, controls intensity of sharpening (0 to 1, default=0.5).
    Returns:
        torch.Tensor, sharpened image normalized [0,1].
    """
    img = img_tensor.detach().cpu().numpy()
    if img.dtype != np.uint8:
        img = (img * 255).astype(np.uint8)
    if img.ndim == 3 and img.shape[0] == 1:
        img = img.squeeze(0)
    elif img.ndim == 3 and img.shape[2] == 1:
        img = img[:, :, 0]

    # Adaptive kernel size based on image dimensions
    if kernel_size is None:
        min_dim = min(img.shape[:2])
        kernel_size = max(3, min(7, min_dim // 50 * 2 + 1))  # Ensure odd, between 3 and 7

    # Create structuring element
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    
    # Apply top-hat transform
    tophat = cv2.morphologyEx(img, cv2.MORPH_TOPHAT, kernel)
    
    # Scale top-hat result and add to original image
    sharpened = cv2.addWeighted(img, 1.0, tophat, sharpening_factor, 0.0)
    
    # Normalize to [0,1]
    sharpened = sharpened.astype(np.float32) / 255.0
    return torch.from_numpy(sharpened).float()


In [ ]:
class BraTS2020_2DSliceDataset(Dataset):
    def __init__(self, csv_path, resize_to=(224,224), only_tumor=False):
        self.df = pd.read_csv(csv_path)
        self.resize = T.Resize(resize_to)
        
        if only_tumor:
            self.df = self.df[self.df['target'] == 1].reset_index(drop=True)
        else:
            self.df = self.df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        file_path = self.df.iloc[index]['slice_path']
        file_path = file_path.replace("../input", "/kaggle/input")
        
    
        with h5py.File(file_path, 'r') as f:
             image = f['image'][:]  # (H*W*4)
             mask = f['mask'][:]   # (H*W*3)

        # Convert to torch tensors and permute to [C, H, W]
        image = torch.from_numpy(image).permute(2, 0, 1).float()
        mask = torch.from_numpy(mask).permute(2, 0, 1).float()

        # Resize to uniform resolution (224x224 or 256x256)
        image = self.resize(image.unsqueeze(0)).squeeze(0)
        mask = self.resize(mask.unsqueeze(0)).squeeze(0)
        image_original = image

        # Apply preprocessing per channel
        image = torch.stack([adaptive_median_normalization(image[i]) for i in range(image.shape[0])])
        image = torch.stack([clahe(image[i]) for i in range(image.shape[0])])
        image = torch.stack([bilateral_filter(image[i]) for i in range(image.shape[0])])
        image = torch.stack([morphological_sharpening(image[i]) for i in range(image.shape[0])])
    
        return image, mask, image_original

In [ ]:
BraTS2020Dataset=BraTS2020_2DSliceDataset(
    csv_path='/kaggle/input/brats2020-training-data/BraTS20 Training Metadata.csv',
)

In [ ]:
# Load one sample
img_processed, mask, img_original = BraTS2020Dataset[55]

# Convert to numpy arrays
img_p = img_processed.numpy()
img_o = img_original.numpy()

modality_names = ['T1', 'T1Gd', 'T2', 'FLAIR']

# Plot comparison for each modality
for i in range(4):
    plt.figure(figsize=(8, 3))
    plt.subplot(1, 2, 1)
    plt.imshow(img_o[i], cmap='gray')
    plt.title(f"Original - {modality_names[i]}")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(img_p[i], cmap='gray')
    plt.title(f"Processed - {modality_names[i]}")
    plt.axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
def normalize_img(img):
    """Normalize image to 0-1 and return float64."""
    img_min = np.min(img)
    img_max = np.max(img)
    if img_max - img_min == 0:
        return np.zeros_like(img, dtype=np.float64)
    img_norm = (img - img_min) / (img_max - img_min)
    return img_norm.astype(np.float64)

In [ ]:
def batch_clahe(images, clip_limit=2.0, tile_grid_size=(16,16)):
    """Apply CLAHE to a batch of images (numpy array of shape [N, H, W])."""
    norm_imgs = np.array([normalize_img(img) for img in images])           # float64 in [0, 1]
    norm_uint8 = (norm_imgs * 255).astype(np.uint8)                        # convert to uint8 for CLAHE
    clahe_imgs = np.array([apply_clahe(img, clip_limit, tile_grid_size) for img in norm_uint8])
    return norm_imgs, clahe_imgs

In [ ]:
def apply_clahe(img, clip_limit=2.0, tile_grid_size=(16, 16)):
    """
    Apply CLAHE to a uint8 grayscale image.
    Expects img in range [0, 255] and dtype=uint8.
    """
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe.apply(img)

In [ ]:
def plot_comparison(norm_imgs, clahe_imgs, modality_names):
    """Plot normalized (float) and CLAHE images side by side with histograms."""
    for i, name in enumerate(modality_names):
        fig, axs = plt.subplots(2, 2, figsize=(16, 16))
        
        # Normalized image
        axs[0, 0].imshow(norm_imgs[i], cmap='gray', vmin=0, vmax=1)
        axs[0, 0].set_title(f"Normalized - {name}")
        axs[0, 0].axis('off')
        
        # CLAHE image
        axs[0, 1].imshow(clahe_imgs[i], cmap='gray', vmin=0, vmax=255)
        axs[0, 1].set_title(f"CLAHE - {name}")
        axs[0, 1].axis('off')
        
        # Histogram for normalized image
        axs[1, 0].hist(norm_imgs[i].flatten(), bins=50, color='gray')
        axs[1, 0].set_title('Histogram Before CLAHE')
        
        # Histogram for CLAHE image
        axs[1, 1].hist(clahe_imgs[i].flatten(), bins=50, color='gray')
        axs[1, 1].set_title('Histogram After CLAHE')
        
        plt.tight_layout()
        plt.show()

# -------------------------------
# Example usage
# -------------------------------
# img_p: numpy array of shape (4, H, W) representing 4 grayscale modalities
modality_names = ['T1', 'T1Gd', 'T2', 'FLAIR']
norm_imgs, clahe_imgs = batch_clahe(img_p, clip_limit=3.0, tile_grid_size=(16, 16))
plot_comparison(norm_imgs, clahe_imgs, modality_names)

# Brain Tumor Segmentation with Fuzzy C-Means + LBP Features + SVM/KNN + PostSegXAI on BraTS2020 Sample 55

In [ ]:
# segmenter.py

import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, Dict, List, Optional

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import log_loss

from skimage.feature import local_binary_pattern
from skimage.morphology import (
    disk, dilation, erosion, binary_erosion, binary_dilation,
    remove_small_holes, remove_small_objects
)
from skimage.measure import label
from skimage.filters import sobel, gabor
from scipy.spatial.distance import directed_hausdorff

try:
    from skimage.segmentation import slic
    _HAS_SLIC = True
except Exception:
    _HAS_SLIC = False


class BrainTumorSegmenter:
    def __init__(
        self,
        c: int = 16,
        m: float = 2.0,
        min_region_size: int = 50,
        max_regions: int = 1200,
        random_state: int = 42,
        use_iterative_cls: bool = True,
        early_stopping_patience: int = 10,
        max_epochs: int = 200,
        batches_per_epoch: int = 32,
        batch_log_every: int = 200,
        sgd_fixed_batch_size: Optional[int] = None,
        region_method: str = "auto",
        slic_segments: int = 400,
        slic_compactness: float = 0.1,
        morph_radius: int = 3,
        min_obj_size: int = 64,
        keep_largest_k: int = 1,
        use_gabor: bool = True,
        use_spatial: bool = True,
        reliability_plot: bool = False,
    ):
        self.c = c
        self.m = m
        self.min_region_size = min_region_size
        self.max_regions = max_regions
        self.random_state = random_state
        self.rng = np.random.default_rng(random_state)

        self.use_iterative_cls = use_iterative_cls
        self.early_stopping_patience = early_stopping_patience
        self.max_epochs = max_epochs
        self.batches_per_epoch = batches_per_epoch
        self.batch_log_every = batch_log_every
        self.sgd_fixed_batch_size = sgd_fixed_batch_size

        self.region_method = region_method
        self.slic_segments = slic_segments
        self.slic_compactness = slic_compactness

        self.morph_radius = morph_radius
        self.min_obj_size = min_obj_size
        self.keep_largest_k = keep_largest_k

        self.use_gabor = use_gabor
        self.use_spatial = use_spatial
        self.reliability_plot = reliability_plot

        self.classifiers: Dict[str, Dict[str, object]] = {}
        self.scalers: Dict[str, StandardScaler] = {}
        self.trained_modalities: Dict[str, bool] = {}
        self.loss_history: Dict[str, Dict[str, List[float]]] = {}
        self.best_model_key: Dict[str, str] = {}
        self.val_curves: Dict[str, Dict[str, float]] = {}
        self.regioner_choice: Dict[str, str] = {}

        self.fused_indices_per_modality: Dict[str, List[int]] = {}
        self.fusion_needed: Dict[str, bool] = {}

        self.modality_names = ['T1', 'T1Gd', 'T2', 'FLAIR']

        self.model_losses: Dict[str, Dict[str, Dict[str, float]]] = {}
        self.training_summary: Dict[str, Dict[str, object]] = {}

    # --- clustering fallback (FCM) ---
    def fuzzy_c_means(self, data: np.ndarray, max_iter: int = 300, tol: float = 1e-5) -> Tuple[np.ndarray, np.ndarray]:
        N = data.shape[0]
        U = self.rng.dirichlet(np.ones(self.c), size=N)
        for _ in range(max_iter):
            U_old = U.copy()
            centers = np.zeros((self.c, data.shape[1]))
            for j in range(self.c):
                um = U[:, j] ** self.m
                centers[j] = np.sum((um[:, None]) * data, axis=0) / (np.sum(um) + 1e-12)
            dist = np.zeros((N, self.c))
            for j in range(self.c):
                diff = data - centers[j]
                dist[:, j] = np.linalg.norm(diff, axis=1)
            dist = np.fmax(dist, 1e-10)
            power = 2.0 / (self.m - 1.0)
            inv = (dist[:, :, None] / dist[:, None, :]) ** power
            U = 1.0 / np.sum(inv, axis=2)
            if np.linalg.norm(U - U_old) < tol:
                break
        return centers, U

    def _make_regions(self, img: np.ndarray, prefer: Optional[str] = None) -> np.ndarray:
        method = (prefer or self.region_method).lower()
        if method == "slic" or (method == "auto" and _HAS_SLIC):
            try:
                labels = slic(
                    img, n_segments=self.slic_segments, compactness=self.slic_compactness,
                    start_label=0, channel_axis=None
                )
            except TypeError:
                labels = slic(img, n_segments=self.slic_segments, compactness=self.slic_compactness, start_label=0)
            return labels.astype(int)
        h, w = img.shape
        _, U = self.fuzzy_c_means(img.reshape(-1, 1).astype(np.float64))
        return np.argmax(U, axis=1).reshape(h, w).astype(int)

    @staticmethod
    def _lbp_hist_on_mask(img: np.ndarray, mask: np.ndarray, P: int = 8, R: float = 1) -> List[float]:
        img = img.astype(np.float32)
        vmin, vmax = float(np.nanmin(img)), float(np.nanmax(img))
        if vmax - vmin < 1e-8:
            norm = np.zeros_like(img, dtype=np.uint8)
        else:
            norm = ((img - vmin) / (vmax - vmin) * 255.0).astype(np.uint8)
        lbp = local_binary_pattern(norm, P, R, method='uniform')
        vals = lbp[mask]
        hist, _ = np.histogram(vals, bins=np.arange(0, P + 3), range=(0, P + 2))
        hist = hist.astype(float); hist /= (hist.sum() + 1e-6)
        return hist.tolist()

    def _region_features(self, img: np.ndarray, region_mask: np.ndarray, extra_modalities: Optional[List[np.ndarray]] = None) -> List[float]:
        feats: List[float] = []
        for R in (1, 2, 3):
            feats += self._lbp_hist_on_mask(img, region_mask, P=8, R=R)
        vals = img[region_mask]
        if vals.size == 0:
            vals = np.array([0.0])
        feats += [
            float(vals.mean()), float(vals.std() + 1e-9),
            float(vals.min()), float(vals.max()),
            float(np.percentile(vals, 10)), float(np.median(vals)), float(np.percentile(vals, 90)),
        ]
        sob = sobel(img)
        feats.append(float(sob[region_mask].mean()))
        if self.use_gabor:
            for freq in (0.1, 0.2):
                for theta in (0, np.pi/4, np.pi/2, 3*np.pi/4):
                    real, imag = gabor(img, frequency=freq, theta=theta)
                    feats.append(float(np.hypot(real, imag)[region_mask].mean()))
        if self.use_spatial:
            rr, cc = np.where(region_mask)
            cy = (rr.mean() / img.shape[0]) if rr.size else 0.0
            cx = (cc.mean() / img.shape[1]) if cc.size else 0.0
            area_frac = region_mask.mean()
            feats += [float(cx), float(cy), float(area_frac)]
        if extra_modalities:
            for mimg in extra_modalities:
                mvals = mimg[region_mask]
                feats += [float(mvals.mean()), float(mvals.std() + 1e-9)]
                feats += self._lbp_hist_on_mask(mimg, region_mask, P=8, R=1)
        return feats

    @staticmethod
    def _to_pos_proba(y_proba) -> np.ndarray:
        proba = np.asarray(y_proba)
        if proba.ndim == 1:
            return proba
        if proba.ndim == 2:
            return proba[:, 1] if proba.shape[1] >= 2 else proba[:, -1]
        raise ValueError("Unexpected probability array shape")

    @staticmethod
    def _log_loss_safe(y_true, y_prob_pos):
        y_true = np.asarray(y_true).astype(int)
        p1 = np.clip(np.asarray(y_prob_pos).reshape(-1), 1e-9, 1 - 1e-9)
        proba_2 = np.column_stack([1 - p1, p1])
        return log_loss(y_true, proba_2, eps=1e-9, labels=[0, 1])

    @staticmethod
    def _smooth_binary(binary_mask: np.ndarray, radius: int, min_obj: int, keep_k: int) -> np.ndarray:
        struct = disk(radius)
        closed = erosion(dilation(binary_mask, struct), struct).astype(bool)
        closed = remove_small_holes(closed, area_threshold=radius * radius)
        closed = remove_small_objects(closed, min_size=max(1, min_obj))
        if keep_k is not None and keep_k > 0:
            lbl = label(closed)
            sizes = [(i, (lbl == i).sum()) for i in range(1, lbl.max() + 1)]
            sizes.sort(key=lambda x: x[1], reverse=True)
            keep = {i for i, _ in sizes[:keep_k]}
            closed = np.isin(lbl, list(keep))
        return closed.astype(np.uint8)

    class _ConstantProbaModel:
        def __init__(self, p: float):
            self.p = float(np.clip(p, 0.0, 1.0))
        def predict_proba(self, X):
            n = len(X)
            return np.column_stack([np.full(n, 1 - self.p), np.full(n, self.p)])

    class _SoftEnsemble:
        def __init__(self, models: List[object]):
            self.models = [m for m in models if m is not None]
        def predict_proba(self, X):
            if not self.models:
                p = np.full(len(X), 0.5)
            else:
                ps = [BrainTumorSegmenter._to_pos_proba(m.predict_proba(X)) for m in self.models]
                p = np.clip(np.mean(ps, axis=0), 1e-6, 1 - 1e-6)
            return np.column_stack([1 - p, p])

    def _build_features_from_labels(self, img: np.ndarray, mask: np.ndarray, labels: np.ndarray,
                                    img_stack: Optional[np.ndarray], fuse_idx: Optional[List[int]]):
        features, targets, region_ids = [], [], []
        for rid in np.unique(labels):
            region_mask = (labels == rid)
            if np.sum(region_mask) < self.min_region_size:
                continue
            extra = None
            if img_stack is not None and fuse_idx:
                extra = [img_stack[j] for j in fuse_idx]
            feats = self._region_features(img, region_mask, extra_modalities=extra)
            features.append(feats)
            targets.append(1 if np.any(mask[region_mask]) else 0)
            region_ids.append(rid)
            if len(features) >= self.max_regions:
                break
        return np.asarray(features), np.asarray(targets), np.asarray(region_ids)

    def process_modality(
        self,
        img: np.ndarray,
        mask: np.ndarray,
        modality_name: str,
        val_size: float = 0.20,
        test_size: float = 0.15,
        img_stack: Optional[np.ndarray] = None,
        fuse_idx: Optional[List[int]] = None,
    ):
        self.trained_modalities[modality_name] = False

        fuse_idx = list(fuse_idx) if fuse_idx is not None else []
        self.fused_indices_per_modality[modality_name] = fuse_idx
        self.fusion_needed[modality_name] = (img_stack is not None and len(fuse_idx) > 0)

        methods = [self.region_method] if self.region_method != "auto" else (["slic"] if _HAS_SLIC else []) + ["fcm"]
        best_val = np.inf
        best_payload = None
        best_method = None

        for method in methods:
            labels = self._make_regions(img, prefer=method)
            feats, targs, _rids = self._build_features_from_labels(img, mask, labels, img_stack=img_stack, fuse_idx=fuse_idx)
            if feats.size == 0:
                continue

            # test split first
            if len(np.unique(targs)) == 2 and np.bincount(targs, minlength=2).min() >= 2:
                X_temp, X_te, y_temp, y_te = train_test_split(feats, targs, test_size=test_size, stratify=targs, random_state=self.random_state)
            else:
                X_temp, X_te, y_temp, y_te = train_test_split(feats, targs, test_size=test_size, random_state=self.random_state)

            rel_val = val_size / max(1e-9, (1.0 - test_size))
            if len(np.unique(y_temp)) == 2 and np.bincount(y_temp, minlength=2).min() >= 2:
                X_tr, X_val, y_tr, y_val = train_test_split(X_temp, y_temp, test_size=rel_val, stratify=y_temp, random_state=self.random_state)
            else:
                X_tr, X_val, y_tr, y_val = train_test_split(X_temp, y_temp, test_size=rel_val, random_state=self.random_state)

            scaler = StandardScaler().fit(X_tr)
            Xtr = scaler.transform(X_tr); Xva = scaler.transform(X_val); Xte = scaler.transform(X_te)

            # --- histories ---
            loss_hist_tr, loss_hist_va, loss_hist_te, batch_losses = [], [], [], []
            candidates: Dict[str, Tuple[object, float, float, float]] = {}
            sgd_meta = {'best_epoch': None, 'epochs_run': None, 'early_stopped': None}

            # --- SGD with per-epoch train/val/test logging ---
            if self.use_iterative_cls and len(np.unique(y_tr)) == 2:
                sgd = SGDClassifier(
                    loss='log_loss', penalty='l2', alpha=1e-4,
                    learning_rate='optimal', random_state=self.random_state,
                    warm_start=True, max_iter=1, tol=None
                )
                sgd.partial_fit(Xtr, y_tr, classes=np.array([0, 1]))

                n_pos, n_neg = np.sum(y_tr == 1), np.sum(y_tr == 0)
                w_pos = len(y_tr) / (2.0 * max(1, n_pos))
                w_neg = len(y_tr) / (2.0 * max(1, n_neg))

                best_va = np.inf; best_state = None; patience = self.early_stopping_patience
                best_epoch_idx = 0; early_stop_triggered = False

                use_fixed = self.sgd_fixed_batch_size is not None and self.sgd_fixed_batch_size >= 1
                batch_size = int(self.sgd_fixed_batch_size) if use_fixed else max(1, len(Xtr) // max(1, self.batches_per_epoch))
                num_batches = max(1, self.batches_per_epoch) if use_fixed else int(np.ceil(len(Xtr) / batch_size))

                for epoch in range(1, self.max_epochs + 1):
                    perm = self.rng.permutation(len(Xtr))
                    Xs, Ys = Xtr[perm], y_tr[perm]
                    for b in range(num_batches):
                        if use_fixed:
                            idx = self.rng.integers(0, len(Xtr), size=batch_size, dtype=np.int64)
                            Xb, Yb = Xtr[idx], y_tr[idx]
                        else:
                            s, e = b * batch_size, (b + 1) * batch_size
                            Xb, Yb = Xs[s:e], Ys[s:e]
                            if len(Xb) == 0: continue
                        sw = np.where(Yb == 1, w_pos, w_neg)
                        sgd.partial_fit(Xb, Yb, sample_weight=sw)
                        pb = self._to_pos_proba(sgd.predict_proba(Xb))
                        batch_losses.append(self._log_loss_safe(Yb, pb))

                        if (b + 1) % max(1, self.batch_log_every) == 0:
                            print(f"[{modality_name}/{method}] Epoch {epoch:>3} | Batch {b+1:>4}/{num_batches} | size={len(Xb)} | loss={batch_losses[-1]:.6f}")

                    p_tr = self._to_pos_proba(sgd.predict_proba(Xtr))
                    p_va = self._to_pos_proba(sgd.predict_proba(Xva))
                    p_te = self._to_pos_proba(sgd.predict_proba(Xte))
                    tr_ll = self._log_loss_safe(y_tr, p_tr)
                    va_ll = self._log_loss_safe(y_val, p_va)
                    te_ll = self._log_loss_safe(y_te, p_te)
                    loss_hist_tr.append(tr_ll); loss_hist_va.append(va_ll); loss_hist_te.append(te_ll)

                    if va_ll + 1e-6 < best_va:
                        best_va = va_ll
                        best_state = sgd.__dict__.copy()
                        best_epoch_idx = epoch
                        patience = self.early_stopping_patience
                    else:
                        patience -= 1
                        if patience <= 0:
                            early_stop_triggered = True
                            break

                if best_state is not None:
                    sgd.__dict__ = best_state
                sgd_meta.update({'best_epoch': best_epoch_idx, 'epochs_run': epoch, 'early_stopped': early_stop_triggered})

                sgd_train_ll = self._log_loss_safe(y_tr, self._to_pos_proba(sgd.predict_proba(Xtr)))
                sgd_val_ll   = self._log_loss_safe(y_val, self._to_pos_proba(sgd.predict_proba(Xva)))
                sgd_test_ll  = self._log_loss_safe(y_te, self._to_pos_proba(sgd.predict_proba(Xte)))
                candidates['sgd'] = (sgd, sgd_val_ll, sgd_train_ll, sgd_test_ll)

            # --- SVM ---
            if len(np.unique(y_tr)) == 2:
                svm_best, svm_best_va, svm_best_tr, svm_best_te = None, np.inf, np.inf, np.inf
                for C in (0.5, 1.0, 2.0):
                    for gamma in ('scale', 'auto'):
                        svm = SVC(kernel='rbf', C=C, gamma=gamma, probability=True, class_weight='balanced', random_state=self.random_state)
                        svm.fit(Xtr, y_tr)
                        va_ll = self._log_loss_safe(y_val, self._to_pos_proba(svm.predict_proba(Xva)))
                        tr_ll = self._log_loss_safe(y_tr, self._to_pos_proba(svm.predict_proba(Xtr)))
                        te_ll = self._log_loss_safe(y_te, self._to_pos_proba(svm.predict_proba(Xte)))
                        if va_ll < svm_best_va:
                            svm_best_va, svm_best_tr, svm_best_te, svm_best = va_ll, tr_ll, te_ll, svm
                if svm_best is not None:
                    candidates['svm'] = (svm_best, svm_best_va, svm_best_tr, svm_best_te)

                # --- KNN ---
                knn_best, knn_best_va, knn_best_tr, knn_best_te = None, np.inf, np.inf, np.inf
                for k in (3, 5, 7):
                    knn = KNeighborsClassifier(n_neighbors=k, weights='distance')
                    knn.fit(Xtr, y_tr)
                    va_ll = self._log_loss_safe(y_val, self._to_pos_proba(knn.predict_proba(Xva)))
                    tr_ll = self._log_loss_safe(y_tr, self._to_pos_proba(knn.predict_proba(Xtr)))
                    te_ll = self._log_loss_safe(y_te, self._to_pos_proba(knn.predict_proba(Xte)))
                    if va_ll < knn_best_va:
                        knn_best_va, knn_best_tr, knn_best_te, knn_best = va_ll, tr_ll, te_ll, knn
                if knn_best is not None:
                    candidates['knn'] = (knn_best, knn_best_va, knn_best_tr, knn_best_te)
            else:
                p_pos = float(np.mean(y_tr)) if len(y_tr) > 0 else 0.0
                const_model = self._ConstantProbaModel(p_pos)
                const_val_ll = self._log_loss_safe(y_val, np.full_like(y_val, p_pos, dtype=float))
                const_tr_ll  = self._log_loss_safe(y_tr, np.full_like(y_tr,  p_pos, dtype=float))
                const_te_ll  = self._log_loss_safe(y_te, np.full_like(y_te,  p_pos, dtype=float))
                candidates['constant'] = (const_model, const_val_ll, const_tr_ll, const_te_ll)

            model_list = [v[0] for k, v in candidates.items() if k in ('sgd', 'svm', 'knn') and v[0] is not None]
            if len(model_list) >= 2:
                ens = BrainTumorSegmenter._SoftEnsemble(model_list)
                ens_va = self._log_loss_safe(y_val, self._to_pos_proba(ens.predict_proba(Xva)))
                ens_tr = self._log_loss_safe(y_tr, self._to_pos_proba(ens.predict_proba(Xtr)))
                ens_te = self._log_loss_safe(y_te, self._to_pos_proba(ens.predict_proba(Xte)))
                candidates['ensemble'] = (ens, ens_va, ens_tr, ens_te)

            best_key = min(candidates, key=lambda k: candidates[k][1])
            if candidates[best_key][1] < best_val:
                best_val = candidates[best_key][1]
                best_payload = {
                    'scaler': scaler,
                    'candidates': candidates,
                    'best_key': best_key,
                    'loss_hist_tr': loss_hist_tr,
                    'loss_hist_va': loss_hist_va,
                    'loss_hist_te': loss_hist_te,
                    'batch_losses': batch_losses,
                    'Xva': Xva, 'y_val': y_val,
                    'sgd_meta': sgd_meta
                }
                best_method = method

        if best_payload is None:
            print(f"[{modality_name}] No usable regions; skipping.")
            return

        self.scalers[modality_name] = best_payload['scaler']
        self.classifiers[modality_name] = {k: v[0] for k, v in best_payload['candidates'].items()}
        self.best_model_key[modality_name] = best_payload['best_key']
        self.trained_modalities[modality_name] = True

        self.loss_history[modality_name] = {
            'train': best_payload['loss_hist_tr'],
            'val':   best_payload['loss_hist_va'],
            'test':  best_payload['loss_hist_te'],
            'batch': best_payload['batch_losses'],
        }
        self.model_losses[modality_name] = {
            k: {'val': float(v[1]), 'train': float(v[2]), 'test': float(v[3])}
            for k, v in best_payload['candidates'].items()
        }
        self.val_curves[modality_name] = {k: float(v[1]) for k, v in best_payload['candidates'].items()}
        self.regioner_choice[modality_name] = best_method

        bm = self.best_model_key[modality_name]
        self.training_summary[modality_name] = {
            'regioner': best_method,
            'best_model_key': bm,
            'best_epoch': (best_payload.get('sgd_meta', {}) or {}).get('best_epoch', None),
            'epochs_run': (best_payload.get('sgd_meta', {}) or {}).get('epochs_run', None),
            'early_stopped': (best_payload.get('sgd_meta', {}) or {}).get('early_stopped', None),
            'best_val_loss': self.model_losses[modality_name][bm]['val'],
            'best_train_loss': self.model_losses[modality_name][bm]['train'],
            'best_test_loss': self.model_losses[modality_name][bm]['test'],
        }

        print(f"[{modality_name}] Best regioner: {best_method.upper()} | Best model: {bm.upper()} "
              f"(trainLL={self.training_summary[modality_name]['best_train_loss']:.6f}, "
              f"valLL={self.training_summary[modality_name]['best_val_loss']:.6f}, "
              f"testLL={self.training_summary[modality_name]['best_test_loss']:.6f})")

        if bm == 'sgd' and self.training_summary[modality_name]['best_epoch'] is not None:
            print(f"[{modality_name}/SGD] epochs_run={self.training_summary[modality_name]['epochs_run']} | "
                  f"best_epoch={self.training_summary[modality_name]['best_epoch']} | "
                  f"early_stopped={'Yes' if self.training_summary[modality_name]['early_stopped'] else 'No'}")

        if self.use_iterative_cls and len(best_payload['loss_hist_tr']) > 0:
            plt.figure(figsize=(6, 4))
            plt.plot(best_payload['loss_hist_tr'], label='train')
            plt.plot(best_payload['loss_hist_va'], label='val')
            if len(best_payload['loss_hist_te']) > 0:
                plt.plot(best_payload['loss_hist_te'], label='test')
            plt.xlabel('epoch'); plt.ylabel('log-loss'); plt.title(f'{modality_name} - SGD loss (train/val/test)')
            plt.legend(); plt.tight_layout(); plt.show()

        if self.use_iterative_cls and len(best_payload['batch_losses']) > 0:
            plt.figure(figsize=(6, 3))
            plt.plot(best_payload['batch_losses'])
            plt.xlabel('batch updates'); plt.ylabel('batch log-loss'); plt.title(f'{modality_name} - Batch Train Loss')
            plt.tight_layout(); plt.show()

        if self.reliability_plot and self.best_model_key[modality_name] not in ('constant',):
            model = self.classifiers[modality_name][self.best_model_key[modality_name]]
            p_va = self._to_pos_proba(model.predict_proba(best_payload['Xva']))
            self._plot_reliability_curve(best_payload['y_val'], p_va, title=f"{modality_name} - Reliability ({self.best_model_key[modality_name]})")

        self.plot_segmentation_results(img, modality_name, img_stack=img_stack)

    # --- Grad-CAM-like region attribution for linear (SGD) model ---
    def _gradcam_like_overlay(self, img: np.ndarray, modality_name: str, region_map: np.ndarray, img_stack: Optional[np.ndarray] = None) -> Optional[np.ndarray]:
        best_key = self.best_model_key.get(modality_name, 'svm')
        if best_key != 'sgd':
            return None
        fuse_idx = self.fused_indices_per_modality.get(modality_name, [])
        regions = sorted([r for r in np.unique(region_map) if r > 0])
        feats_list = []
        for r in regions:
            rmask = (region_map == r)
            extra = None
            if img_stack is not None and self.fusion_needed.get(modality_name, False) and fuse_idx:
                extra = [img_stack[j] for j in fuse_idx]
            feats_list.append(self._region_features(img, rmask, extra_modalities=extra))
        X = np.asarray(feats_list)
        scaler = self.scalers[modality_name]
        Xs = scaler.transform(X)
        model = self.classifiers[modality_name][best_key]
        coef = getattr(model, "coef_", None)
        intercept = getattr(model, "intercept_", None)
        if coef is None or intercept is None:
            return None
        linear_score = (Xs @ coef.reshape(-1, 1)).ravel() + float(intercept)
        s_min, s_max = float(np.min(linear_score)), float(np.max(linear_score))
        if abs(s_max - s_min) < 1e-12:
            norm_scores = np.zeros_like(linear_score, dtype=float)
        else:
            norm_scores = (linear_score - s_min) / (s_max - s_min)
        cam = np.zeros_like(img, dtype=float)
        for val, r in zip(norm_scores, regions):
            cam[region_map == r] = val
        return cam

    # --- inference maps ---
    def _region_conf_maps(self, img: np.ndarray, modality_name: str, img_stack: Optional[np.ndarray] = None) -> Tuple[np.ndarray, Optional[np.ndarray], np.ndarray]:
        fuse_idx = self.fused_indices_per_modality.get(modality_name, [])
        if self.fusion_needed.get(modality_name, False) and img_stack is None:
            raise ValueError(f"{modality_name}: trained with fused features; provide img_stack=...")
        labels = self._make_regions(img, prefer=self.regioner_choice.get(modality_name, self.region_method))
        features: List[List[float]] = []
        region_map = np.zeros_like(labels, dtype=int)
        region_ids: List[int] = []
        rid = 1
        for r in np.unique(labels):
            region_mask = (labels == r)
            if np.sum(region_mask) < self.min_region_size:
                continue
            extra = None
            if img_stack is not None and fuse_idx:
                extra = [img_stack[j] for j in fuse_idx]
            feats = self._region_features(img, region_mask, extra_modalities=extra)
            features.append(feats)
            region_map[region_mask] = rid
            region_ids.append(rid)
            rid += 1
        if len(features) == 0:
            return np.zeros_like(img, dtype=float), None, region_map
        X = np.asarray(features)
        scaler = self.scalers[modality_name]
        if X.shape[1] != scaler.n_features_in_:
            raise ValueError(f"{modality_name}: feature size mismatch ({X.shape[1]} vs {scaler.n_features_in_}). Ensure img_stack matches fusion used in training.")
        X_scaled = scaler.transform(X)
        best_key = self.best_model_key.get(modality_name, 'svm')
        model = self.classifiers[modality_name][best_key]
        probs = self._to_pos_proba(model.predict_proba(X_scaled))
        conf_map = np.zeros_like(img, dtype=float)
        for i, r in enumerate(region_ids):
            conf_map[region_map == r] = probs[i]
        conf_map_alt = None
        if best_key != 'svm' and 'svm' in self.classifiers[modality_name]:
            probs_svm = self._to_pos_proba(self.classifiers[modality_name]['svm'].predict_proba(X_scaled))
            conf_map_alt = np.zeros_like(img, dtype=float)
            for i, r in enumerate(region_ids):
                conf_map_alt[region_map == r] = probs_svm[i]
        return conf_map, conf_map_alt, region_map

    def plot_segmentation_results(self, img: np.ndarray, modality_name: str, img_stack: Optional[np.ndarray] = None):
        conf_map, conf_map_alt, region_map = self._region_conf_maps(img, modality_name, img_stack=img_stack)
        binary_mask = (conf_map > 0.5).astype(np.uint8)
        smooth = self._smooth_binary(binary_mask, self.morph_radius, self.min_obj_size, self.keep_largest_k)
        plt.figure(figsize=(12, 10))
        plt.subplot(2, 2, 1); plt.imshow(img, cmap='gray'); plt.imshow(smooth, alpha=0.4, cmap='Reds'); plt.title(f"{modality_name} - Smoothed Mask (best)")
        plt.subplot(2, 2, 2); plt.imshow(conf_map, cmap='hot'); plt.title("Confidence Heatmap (best)")
        if conf_map_alt is not None:
            plt.subplot(2, 2, 3); plt.imshow(conf_map_alt, cmap='hot'); plt.title("Confidence Heatmap (SVM)")
        cam = self._gradcam_like_overlay(img, modality_name, region_map, img_stack=img_stack)
        plt.subplot(2, 2, 4); plt.imshow(img, cmap='gray')
        if cam is not None:
            plt.imshow(cam, alpha=0.5, cmap='jet'); plt.title("Grad-CAM-like Overlay (best)")
        else:
            plt.imshow(conf_map, alpha=0.5, cmap='hot'); plt.title("Overlay (confidence)")
        plt.tight_layout(); plt.show()

    def run_postxai(self, img: np.ndarray, mask: np.ndarray, modality_name: str,
                    thresh: float = 0.5, thresh_mode: str = "fixed", img_stack: Optional[np.ndarray] = None) -> Dict[str, object]:
        if not self.trained_modalities.get(modality_name, False):
            print(f"PostSegXAI skipped: {modality_name} not trained.")
            return {}
        conf_map, _, region_map = self._region_conf_maps(img, modality_name, img_stack=img_stack)
        if thresh_mode == "otsu":
            try:
                from skimage.filters import threshold_otsu
                t = float(threshold_otsu(conf_map[np.isfinite(conf_map)]))
            except Exception:
                t = float(thresh)
        elif thresh_mode == "max_dice":
            t = self._best_threshold_by_dice(mask.astype(bool), conf_map)
        else:
            t = float(thresh)
        binary_mask = (conf_map > t).astype(np.uint8)
        smooth = self._smooth_binary(binary_mask, self.morph_radius, self.min_obj_size, self.keep_largest_k)
        uncertainty = (conf_map > (t - 0.1)) & (conf_map < (t + 0.1))
        metrics = self.compute_metrics(mask.astype(bool), smooth.astype(bool))
        bm = self.best_model_key.get(modality_name, 'svm')
        regioner = self.regioner_choice.get(modality_name, self.region_method)
        train_ll = val_ll = test_ll = None
        if modality_name in self.model_losses and bm in self.model_losses[modality_name]:
            train_ll = self.model_losses[modality_name][bm]['train']
            val_ll   = self.model_losses[modality_name][bm]['val']
            test_ll  = self.model_losses[modality_name][bm]['test']
        cam = self._gradcam_like_overlay(img, modality_name, region_map, img_stack=img_stack)
        explanation_type = "gradcam_like" if cam is not None else "confidence_overlay"

        plt.figure(figsize=(16, 16))
        plt.subplot(2, 2, 1); plt.imshow(img, cmap='gray'); plt.imshow(smooth, alpha=0.4, cmap='Reds'); plt.title(f"{modality_name} - Smoothed Mask (thr={t:.3f})")
        plt.subplot(2, 2, 2); plt.imshow(conf_map, cmap='hot'); plt.title("Confidence Heatmap")
        plt.subplot(2, 2, 3); plt.imshow(uncertainty.astype(np.uint8), cmap='hot'); plt.title("Uncertainty (±0.1)")
        plt.subplot(2, 2, 4); plt.imshow(img, cmap='gray')
        if cam is not None:
            plt.imshow(cam, alpha=0.5, cmap='jet'); plt.title("Grad-CAM-like Overlay")
        else:
            plt.imshow(conf_map, alpha=0.5, cmap='hot'); plt.title("Overlay (confidence)")
        plt.tight_layout(); plt.show()

        print(f"[{modality_name}] thr={t:.3f} ({thresh_mode}) | "
              f"Acc={metrics['accuracy']:.4f} DSC={metrics['dsc']:.4f} IoU={metrics['iou']:.4f} "
              f"P={metrics['precision']:.4f} R={metrics['recall']:.4f} BF1={metrics['boundary_f1']:.4f} "
              f"{'HD=' + format(metrics['hausdorff'], '.2f') if metrics.get('hausdorff') is not None else ''} | "
              f"Overlay={metrics['overlay_quality']} | "
              f"BestModel={bm.upper()} (trainLL={train_ll if train_ll is not None else float('nan'):.4f}, "
              f"valLL={val_ll if val_ll is not None else float('nan'):.4f}, "
              f"testLL={test_ll if test_ll is not None else float('nan'):.4f}) | "
              f"Explanation={explanation_type}")

        return {
            'threshold': float(t),
            'tumor_size_px': int(np.sum(smooth)),
            'uncertainty_area_px': int(np.sum(uncertainty)),
            'best_model': bm,
            'regioner': regioner,
            'train_log_loss': float(train_ll) if train_ll is not None else None,
            'val_log_loss': float(val_ll)   if val_ll   is not None else None,
            'test_log_loss': float(test_ll) if test_ll  is not None else None,
            'explanation_available': cam is not None,
            'explanation_type': explanation_type,
            **metrics
        }

    def _best_threshold_by_dice(self, gt: np.ndarray, conf_map: np.ndarray) -> float:
        gt = gt.astype(bool)
        if not np.any(gt): return 0.5
        ts = np.linspace(0.05, 0.95, 37)
        best_t, best_d = 0.5, -1.0
        for t in ts:
            pred = self._smooth_binary((conf_map > t).astype(np.uint8), self.morph_radius, self.min_obj_size, self.keep_largest_k).astype(bool)
            d = self.compute_metrics(gt, pred, compute_hd=False)['dsc']
            if d > best_d + 1e-9 or (abs(d - best_d) < 1e-9 and t < best_t):
                best_d, best_t = d, t
        return float(best_t)

    @staticmethod
    def _boundary_f1(gt: np.ndarray, pred: np.ndarray, tol: int = 2) -> float:
        gt = gt.astype(bool); pred = pred.astype(bool)
        gt_edge = np.logical_xor(gt, binary_erosion(gt))
        pr_edge = np.logical_xor(pred, binary_erosion(pred))
        gt_band = binary_dilation(gt_edge, disk(tol))
        pr_band = binary_dilation(pr_edge, disk(tol))
        tp_p = np.logical_and(pr_edge, gt_band).sum()
        prec = tp_p / (pr_edge.sum() + 1e-9)
        tp_r = np.logical_and(gt_edge, pr_band).sum()
        rec = tp_r / (gt_edge.sum() + 1e-9)
        return 0.0 if (prec + rec) == 0 else 2 * prec * rec / (prec + rec)

    @staticmethod
    def _overlay_quality_label(dsc: float, bf1: float) -> str:
        score = 0.7 * dsc + 0.3 * bf1
        if score >= 0.85: return "Excellent"
        if score >= 0.75: return "Good"
        if score >= 0.60: return "Fair"
        return "Poor"

    @staticmethod
    def compute_metrics(gt: np.ndarray, pred: np.ndarray, compute_hd: bool = True) -> Dict[str, Optional[float]]:
        gt = gt.astype(bool); pred = pred.astype(bool)
        tp = np.logical_and(gt, pred).sum()
        fp = np.logical_and(~gt, pred).sum()
        fn = np.logical_and(gt, ~pred).sum()
        tn = np.logical_and(~gt, ~pred).sum()
        total = tp + fp + fn + tn + 1e-9
        accuracy = (tp + tn) / total
        precision = tp / (tp + fp + 1e-9)
        recall = tp / (tp + fn + 1e-9)
        iou = tp / (tp + fp + fn + 1e-9)
        dsc = 2 * tp / (2 * tp + fp + fn + 1e-9)
        boundary_f1 = BrainTumorSegmenter._boundary_f1(gt, pred, tol=2)
        overlay_quality = BrainTumorSegmenter._overlay_quality_label(dsc, boundary_f1)
        hd_val = None
        if compute_hd and gt.any() and pred.any():
            gt_pts, pr_pts = np.argwhere(gt), np.argwhere(pred)
            if len(gt_pts) > 0 and len(pr_pts) > 0:
                hd1 = directed_hausdorff(gt_pts, pr_pts)[0]
                hd2 = directed_hausdorff(pr_pts, gt_pts)[0]
                hd_val = float(max(hd1, hd2))
        return {
            'accuracy': float(accuracy), 'precision': float(precision), 'recall': float(recall),
            'iou': float(iou), 'dsc': float(dsc), 'boundary_f1': float(boundary_f1),
            'overlay_quality': overlay_quality, 'hausdorff': hd_val,
        }

    @staticmethod
    def _plot_reliability_curve(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 8, title: str = "Reliability"):
        y_true = np.asarray(y_true).astype(int)
        y_prob = np.asarray(y_prob).astype(float)
        order = np.argsort(y_prob); y_prob = y_prob[order]; y_true = y_true[order]
        bins = np.linspace(0.0, 1.0, n_bins + 1)
        accs, confs = [], []
        for i in range(n_bins):
            lo, hi = bins[i], bins[i + 1]
            sel = (y_prob >= lo) & (y_prob < hi) if i < n_bins - 1 else (y_prob >= lo) & (y_prob <= hi)
            if sel.sum() == 0: continue
            accs.append(y_true[sel].mean()); confs.append(y_prob[sel].mean())
        plt.figure(figsize=(4.5, 4.5))
        plt.plot([0, 1], [0, 1], linestyle='--', label='Perfect')
        if len(accs) > 0: plt.plot(confs, accs, marker='o', label='Model')
        plt.xlabel('Confidence'); plt.ylabel('Accuracy')
        from sklearn.metrics import brier_score_loss as _brier
        plt.title(f"{title}\nBrier={_brier(y_true, y_prob):.4f}")
        plt.legend(); plt.tight_layout(); plt.show()

    def get_loss_history(self, modality_name: str) -> Dict[str, object]:
        return {
            'epoch_train': list(self.loss_history.get(modality_name, {}).get('train', [])),
            'epoch_val':   list(self.loss_history.get(modality_name, {}).get('val',   [])),
            'epoch_test':  list(self.loss_history.get(modality_name, {}).get('test',  [])),
            'batch_train': list(self.loss_history.get(modality_name, {}).get('batch', [])),
            'model_losses': self.model_losses.get(modality_name, {}),
            'summary': self.training_summary.get(modality_name, {})
        }

    def plot_training_curves(self, modality_name: str, show_batch: bool = True):
        hist = self.loss_history.get(modality_name, {})
        if hist.get('train') and hist.get('val'):
            plt.figure(figsize=(6, 4))
            plt.plot(hist['train'], label='train')
            plt.plot(hist['val'],   label='val')
            if hist.get('test'):
                plt.plot(hist['test'], label='test')
            plt.xlabel('epoch'); plt.ylabel('log-loss'); plt.title(f'{modality_name} - SGD loss (train/val/test)')
            plt.legend(); plt.tight_layout(); plt.show()
        if show_batch and hist.get('batch'):
            plt.figure(figsize=(6, 3))
            plt.plot(hist['batch'])
            plt.xlabel('batch updates'); plt.ylabel('batch log-loss'); plt.title(f'{modality_name} - Batch Train Loss (replot)')
            plt.tight_layout(); plt.show()

    # --- dataset drivers (no npz dependency here) ---
    def process_sample(self, dataset, index: int = 55):
        img_processed, mask, img_original = dataset[index]
        img_o = img_original.numpy()
        mask_np = mask.numpy()[0]
        for i in range(4):
            modality = self.modality_names[i]
            print(f"\nProcessing modality: {modality}")
            self.process_modality(img_o[i], mask_np, modality, img_stack=None, fuse_idx=[])

    def process_sample_fused(self, dataset, index: int = 55):
        img_processed, mask, img_original = dataset[index]
        img_o = img_original.numpy()
        mask_np = mask.numpy()[0]
        for i in range(4):
            fuse_idx = [j for j in range(4) if j != i]
            modality = self.modality_names[i]
            print(f"\nProcessing (fused) modality: {modality} | fuse from {[self.modality_names[j] for j in fuse_idx]}")
            self.process_modality(img_o[i], mask_np, modality, img_stack=img_o, fuse_idx=fuse_idx)

    def run_postxai_all(self, dataset, index: int = 55, thresh: float = 0.5, thresh_mode: str = "fixed") -> Dict[str, Dict[str, object]]:
        img_processed, mask, img_original = dataset[index]
        img_o = img_original.numpy()
        mask_np = mask.numpy()[0]
        reports: Dict[str, Dict[str, object]] = {}
        for i in range(4):
            modality = self.modality_names[i]
            use_stack = img_o if self.fusion_needed.get(modality, False) else None
            print(f"\n==== PostSegXAI for modality: {modality} ====")
            reports[modality] = self.run_postxai(img_o[i], mask_np, modality, thresh=thresh, thresh_mode=thresh_mode, img_stack=use_stack)
        return reports

    def predict_confidence_map(self, img: np.ndarray, modality_name: str, img_stack: Optional[np.ndarray] = None) -> np.ndarray:
        if self.fusion_needed.get(modality_name, False) and img_stack is None:
            raise ValueError(f"{modality_name}: this model was trained with fused features. Provide img_stack=...")
        conf_map, _, _ = self._region_conf_maps(img, modality_name, img_stack=img_stack)
        return conf_map


In [ ]:
# ==== usage (fused features + SLIC superpixels + large-batch SGD) ====

segmenter = BrainTumorSegmenter(
    region_method='slic',          # or 'auto' to fall back to FCM if SLIC isn't available
    slic_segments=400,
    slic_compactness=0.1,
    min_region_size=60,
    morph_radius=3, min_obj_size=64, keep_largest_k=1,

    # training schedule (tune these to scale total compute)
    use_iterative_cls=True,
    sgd_fixed_batch_size=4200,     # <<< forces huge SGD mini-batches (sampling with replacement)
    batches_per_epoch=48,          # number of those big-batch updates per epoch
    max_epochs=800,                # total epochs
    early_stopping_patience=999999,# effectively disable early stopping
    batch_log_every=16,            # reduce log spam during long runs

    reliability_plot=True
)

# Train each modality using fused features from the other three modalities
segmenter.process_sample_fused(BraTS2020Dataset, index=55)

# Run PostSegXAI (automatically passes the image stack if fusion was used)
reports = segmenter.run_postxai_all(BraTS2020Dataset, index=55, thresh_mode="max_dice")

from pprint import pprint
pprint(reports)   # nicely print the per‑modality metrics/report
